# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (FAIR², DOI: 10.71728/senscience.vq4a-28xa).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# For visualization later
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset and show basic metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll enumerate the record sets in the dataset, list their `@id`, and show available fields for each record set.

In [ ]:
# List record sets and associated fields with their @id
print("Available Record Sets (referenced by @id):\n------------------------------")
record_sets = []
for rs in metadata.record_sets:
    print(f"- @id: {rs.id}  | Name: {getattr(rs, 'name', 'N/A')}")
    record_sets.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields (@id and name):")
        for f in rs.fields:
            print(f"    - {f.id} | {getattr(f, 'name', 'N/A')}")
    else:
        print("  No fields listed.")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Let's select the main record set for protein data (replace with the correct `@id` from the above cell; here we use the likely common primary set).

In [ ]:
# Replace this with the actual @id of the main protein record set as displayed above
main_record_set_id = record_sets[0] if record_sets else None

if main_record_set_id is None:
    raise ValueError('No record sets found in this dataset.')

# Extract records from all record sets
dataframes = {}
for rsid in record_sets:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df

print(f"Columns in {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's explore numeric features such as molecular weight (MW), abundance, peptide count, etc. We'll filter on a numeric field, normalize, and analyze distributions. 

> For demonstration, we select a numeric field that exists in this record set. Adjust the field name as appropriate to your data after inspecting the list of columns.

In [ ]:
# Select numeric field (e.g., 'MW', 'abundance', 'peptide_count').
# Replace these with actual column names from dataframes[main_record_set_id] as necessary.

numeric_fields = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype in ['float64', 'int64']]
if not numeric_fields:
    # If there are no numeric fields by dtype, try to infer, e.g. MW or abundance is often float but might be string
    for col in dataframes[main_record_set_id].columns:
        try:
            dataframes[main_record_set_id][col] = pd.to_numeric(dataframes[main_record_set_id][col])
        except Exception:
            pass
    numeric_fields = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype in ['float64', 'int64']]

if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Selected numeric field: {numeric_field}")
else:
    # Demonstration fallback
    raise ValueError('No numeric fields available in this record set.')

# Filtering: remove records with numeric_field below a threshold
threshold = dataframes[main_record_set_id][numeric_field].mean()
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization (Z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical key (e.g., 'accession' or 'sample_id' if exists)
candidate_group_fields = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype == 'object']
group_field = candidate_group_fields[0] if candidate_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field and (if grouping above succeeded) a boxplot/grouped mean visualization.

In [ ]:
# Distribution plot
plt.figure(figsize=(8,4))
sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f'Distribution of {numeric_field}')
plt.show()

if group_field:
    # Show top N groups by count
    top_cats = dataframes[main_record_set_id][group_field].value_counts().head(10).index.tolist()
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id][dataframes[main_record_set_id][group_field].isin(top_cats)])
    plt.title(f'{numeric_field} by {group_field} (Top 10 categories)')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded a FAIR²-compliant Croissant dataset describing mass spectrometry analysis of extracellular vesicles, explored its schema, loaded tabular protein measurement data, performed basic numeric data filtering and normalization, and visualized key distributions.

You can extend this analysis with hypothesis testing, more advanced visualizations, or downstream modeling tasks using the structured information provided with each record set and field via the Croissant standard.

Feel free to rerun this notebook and modify it for other Croissant datasets or additional analyses!